In [34]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [35]:
df = pd.read_csv('cleaned_data.csv')
df.sample()

,age,RevolvingUtilizationOfUnsecuredLines,NumberOfTime30-59DaysPastDueNotWorse,NumberOfTime60-89DaysPastDueNotWorse,NumberOfTimes90DaysLate,DebtRatio,DebtRatioMissing,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberRealEstateLoansOrLines,NumberOfDependents,SeriousDlqin2yrs
104347,53,0.051199,0,0,0,0.224362,0,14850.0,7,2,3.0,0


In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134206 entries, 0 to 134205
Data columns (total 12 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   age                                   134206 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  134206 non-null  float64
 2   NumberOfTime30-59DaysPastDueNotWorse  134206 non-null  int64  
 3   NumberOfTime60-89DaysPastDueNotWorse  134206 non-null  int64  
 4   NumberOfTimes90DaysLate               134206 non-null  int64  
 5   DebtRatio                             134206 non-null  float64
 6   DebtRatioMissing                      134206 non-null  int64  
 7   MonthlyIncome                         134206 non-null  float64
 8   NumberOfOpenCreditLinesAndLoans       134206 non-null  int64  
 9   NumberRealEstateLoansOrLines          134206 non-null  int64  
 10  NumberOfDependents                    134206 non-null  float64
 11  

In [37]:
df.drop(['NumberOfDependents','NumberOfTime30-59DaysPastDueNotWorse','NumberOfTime60-89DaysPastDueNotWorse',],axis=1,inplace=True)

In [38]:
df.columns

Index(['age', 'RevolvingUtilizationOfUnsecuredLines',
       'NumberOfTimes90DaysLate', 'DebtRatio', 'DebtRatioMissing',
       'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans',
       'NumberRealEstateLoansOrLines', 'SeriousDlqin2yrs'],
      dtype='object')

In [40]:
df = df[['age', 'RevolvingUtilizationOfUnsecuredLines', 'NumberOfTimes90DaysLate','DebtRatio','DebtRatioMissing',
        'MonthlyIncome','NumberOfOpenCreditLinesAndLoans','NumberRealEstateLoansOrLines','SeriousDlqin2yrs']]

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 134206 entries, 0 to 134205
Data columns (total 9 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   age                                   134206 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  134206 non-null  float64
 2   NumberOfTimes90DaysLate               134206 non-null  int64  
 3   DebtRatio                             134206 non-null  float64
 4   DebtRatioMissing                      134206 non-null  int64  
 5   MonthlyIncome                         134206 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       134206 non-null  int64  
 7   NumberRealEstateLoansOrLines          134206 non-null  int64  
 8   SeriousDlqin2yrs                      134206 non-null  int64  
dtypes: float64(3), int64(6)
memory usage: 9.2 MB


In [43]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [44]:
from sklearn.model_selection import train_test_split

In [45]:
X_train, X_test,y_train,y_test = train_test_split(X,y,test_size=0.25,random_state=42)

In [46]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

In [47]:
from sklearn.model_selection import cross_val_score

In [48]:
import optuna

In [49]:
from imblearn.over_sampling import SMOTENC

In [50]:
X_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 100654 entries, 83638 to 121958
Data columns (total 8 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   age                                   100654 non-null  int64  
 1   RevolvingUtilizationOfUnsecuredLines  100654 non-null  float64
 2   NumberOfTimes90DaysLate               100654 non-null  int64  
 3   DebtRatio                             100654 non-null  float64
 4   DebtRatioMissing                      100654 non-null  int64  
 5   MonthlyIncome                         100654 non-null  float64
 6   NumberOfOpenCreditLinesAndLoans       100654 non-null  int64  
 7   NumberRealEstateLoansOrLines          100654 non-null  int64  
dtypes: float64(3), int64(5)
memory usage: 6.9 MB


In [51]:
smotenc = SMOTENC(categorical_features=[4],random_state=42)

In [52]:
X_train_resampled, y_train_resampled = smotenc.fit_resample(X_train, y_train)

In [53]:
y_train_resampled.value_counts()

SeriousDlqin2yrs
0    94585
1    94585
Name: count, dtype: int64

In [54]:
def objective(trial):

    classifier_model = trial.suggest_categorical('classifier',['RFC','GB','Logistic'])

    if classifier_model=='Logistic':
        C = trial.suggest_float('logistic_C',0.1,100,log=True)
        class_weight = trial.suggest_categorical('class_weight', [None, 'balanced'])
        solver = trial.suggest_categorical('solver',['lbfgs','liblinear'])
        model = LogisticRegression(C=C,class_weight=class_weight,solver=solver,max_iter=1000)

    elif classifier_model=='RFC':
        n_estimators = trial.suggest_int('estimators',20,150,step=2)
        criterion = trial.suggest_categorical('criterion',['gini', 'entropy', 'log_loss'])
        max_depth = trial.suggest_int('max_depth',3,7)
        min_samples_split = trial.suggest_int('min_samples_split',2,10,step=2)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',2,16,step=2)
        max_features = trial.suggest_categorical('max_features',['sqrt','log2'])
        bootstrap = trial.suggest_categorical('bootstrap',[True,False])

        model = RandomForestClassifier(n_estimators=n_estimators,criterion=criterion,max_depth=max_depth,min_samples_split=min_samples_split,
                                      min_samples_leaf=min_samples_leaf,max_features=max_features,bootstrap=bootstrap,random_state=42)


    elif classifier_model == 'GB':

        n_estimators=trial.suggest_int('n_estimators',50,150,step=2)
        learning_rate=trial.suggest_float('learning_rate',0.01,0.3,log=True)
        min_samples_split = trial.suggest_int('min_samples_split',2,10,step=2)
        min_samples_leaf = trial.suggest_int('min_samples_leaf',2,16,step=2)

        model = GradientBoostingClassifier(n_estimators=n_estimators,learning_rate=learning_rate,
                                           min_samples_split=min_samples_split,min_samples_leaf=min_samples_leaf,random_state=42)

    
    score = cross_val_score(model, X_train_resampled, y_train_resampled, cv=3, scoring='recall').mean()    
    return score  # Return the recall score for Optuna to maximize

In [55]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize recall
study.optimize(objective, n_trials=125)  # Run 125 trials to find the best hyperparameters

[I 2026-08-18 20:55:16,606] A new study created in memory with name: no-name-e05bcad1-f52c-42fd-8fa5-91119844b5ff
C:\Users\bhavy\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
[I 2026-08-18 20:55:32,089] Trial 0 finished with value: 0.7387006354421577 and parameters: {'classifier': 'Logistic', 'logistic_C': 17.37276201099193, 'class_weight': None, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.7387006354421577.
[I 2026-08-18 20:56:35,443] Trial 1 finished with value: 0.823544839696228 and parameters: {'classifier': 'GB', 'n_estimators': 10

KeyboardInterrupt: 

In [56]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.8669133066843804
Best hyperparameters: {'classifier': 'RFC', 'estimators': 108, 'criterion': 'entropy', 'max_depth': 3, 'min_samples_split': 6, 'min_samples_leaf': 12, 'max_features': 'log2', 'bootstrap': False}


In [57]:
Best_model = RandomForestClassifier(n_estimators=108,criterion='entropy',max_depth=3,min_samples_split=6,min_samples_leaf=12,
                                    max_features='log2',bootstrap=False)

In [58]:
from imblearn.pipeline import Pipeline    # Not using sklearn's pipeline, because it does not work with SMOTENC

In [68]:
final_pipe = Pipeline(
    [
     ('SMOTENC',smotenc)
     ('Best Model',Best_model)
    ]
)

In [69]:
final_pipe.fit(X_train, y_train)

Pipeline(steps=[('Best Model',
                 RandomForestClassifier(bootstrap=False, criterion='entropy',
                                        max_depth=3, max_features='log2',
                                        min_samples_leaf=12,
                                        min_samples_split=6,
                                        n_estimators=108))])

In [70]:
y_preds = final_pipe.predict(X_test)

In [71]:
y_preds.shape

(33552,)

In [72]:
X_test.shape

(33552, 8)

In [73]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, recall_score

In [74]:
accuracy_score(y_test,y_preds)

0.9419408679065332

In [75]:
recall_score(y_test,y_preds)

0.0

In [67]:
print('Classification report for Logistic with SMOTENC')
print(classification_report(y_test,y_preds))

Classification report for Logistic with SMOTENC
              precision    recall  f1-score   support

           0       0.98      0.63      0.77     31604
           1       0.12      0.79      0.20      1948

    accuracy                           0.64     33552
   macro avg       0.55      0.71      0.49     33552
weighted avg       0.93      0.64      0.74     33552

